# Counting (Combinatorics)

**Combinatorics** is the art of counting *without* listing everything one by one. "How many passwords are possible?" "How many ways can a committee be chosen?" "What is the probability of a poker hand?" — all are counting questions.

This notebook covers the basic counting rules, **permutations** and **combinations**, the **binomial coefficients** and Pascal's triangle, **inclusion–exclusion**, and the **pigeonhole principle**. Python's `math` and `itertools` modules let us check every formula against a brute-force count.

## The Sum and Product Rules

Two rules underlie almost all counting:

- **Sum rule.** If a task can be done in one of $m$ ways *or* one of $n$ ways (with no overlap), there are $m + n$ ways.
- **Product rule.** If a task is a sequence of two independent choices, the first with $m$ options and the second with $n$, there are $m \cdot n$ ways in total.

For example, a meal with one of $3$ mains *and* one of $4$ desserts can be ordered in $3 \times 4 = 12$ ways.

In [1]:
from itertools import product

mains = ["soup", "salad", "pasta"]
desserts = ["cake", "pie", "ice cream", "fruit"]

meals = list(product(mains, desserts))
print(f"product rule: {len(mains)} x {len(desserts)} = {len(mains) * len(desserts)} meals")
print(f"brute-force count: {len(meals)} meals")
print("first few:", meals[:4])

product rule: 3 x 4 = 12 meals
brute-force count: 12 meals
first few: [('soup', 'cake'), ('soup', 'pie'), ('soup', 'ice cream'), ('soup', 'fruit')]


## Permutations

A **permutation** is an *ordered* arrangement. The number of ways to arrange $k$ of $n$ distinct objects (order matters, no repetition) is

$$P(n, k) = \frac{n!}{(n-k)!} = n \cdot (n-1) \cdots (n-k+1).$$

In particular $P(n, n) = n!$ arranges all $n$ objects. Use `math.perm(n, k)` (and `itertools.permutations` to actually list them).

In [2]:
import math
from itertools import permutations

n, k = 5, 3
print(f"P({n},{k}) = {n}!/{n-k}! =", math.perm(n, k))

# Check against an explicit list of ordered selections of 3 letters from 5.
letters = "ABCDE"
perms = list(permutations(letters, k))
print("brute-force count:", len(perms))
print("a few:", perms[:5])

print("\nArrangements of all 5 letters: P(5,5) =", math.perm(5, 5), "= 5! =", math.factorial(5))

P(5,3) = 5!/2! = 60
brute-force count: 60
a few: [('A', 'B', 'C'), ('A', 'B', 'D'), ('A', 'B', 'E'), ('A', 'C', 'B'), ('A', 'C', 'D')]

Arrangements of all 5 letters: P(5,5) = 120 = 5! = 120


## Combinations

A **combination** is an *unordered* selection. The number of ways to choose $k$ of $n$ distinct objects (order does **not** matter, no repetition) is the **binomial coefficient**

$$C(n, k) = \binom{n}{k} = \frac{n!}{k!\,(n-k)!}.$$

Each unordered choice corresponds to $k!$ ordered ones, so $C(n,k) = P(n,k)/k!$. Use `math.comb(n, k)`.

In [3]:
import math
from itertools import combinations

n, k = 5, 3
print(f"C({n},{k}) =", math.comb(n, k), " and  P(n,k)/k! =", math.perm(n, k) // math.factorial(k))

combos = list(combinations("ABCDE", k))
print("brute-force count:", len(combos))
print("the combinations:", combos)

C(5,3) = 10  and  P(n,k)/k! = 10
brute-force count: 10
the combinations: [('A', 'B', 'C'), ('A', 'B', 'D'), ('A', 'B', 'E'), ('A', 'C', 'D'), ('A', 'C', 'E'), ('A', 'D', 'E'), ('B', 'C', 'D'), ('B', 'C', 'E'), ('B', 'D', 'E'), ('C', 'D', 'E')]


## Binomial Coefficients and Pascal's Triangle

The binomial coefficients arrange themselves into **Pascal's triangle**, where each entry is the sum of the two above it:

$$\binom{n}{k} = \binom{n-1}{k-1} + \binom{n-1}{k} \qquad \text{(Pascal's identity)}.$$

They are also the coefficients in the **binomial theorem**:

$$(x + y)^n = \sum_{k=0}^{n} \binom{n}{k} x^{\,n-k} y^{\,k}.$$

In [4]:
import math

def pascal(n_rows):
    triangle = [[1]]
    for _ in range(1, n_rows):
        prev = triangle[-1]
        triangle.append([1] + [prev[j] + prev[j + 1] for j in range(len(prev) - 1)] + [1])
    return triangle

triangle = pascal(6)
print("Pascal's triangle:")
for row in triangle:
    print("   " + " ".join(f"{v:>3}" for v in row).center(28))

# Each row n is exactly [C(n,0), C(n,1), ..., C(n,n)].
matches = all(row == [math.comb(n, k) for k in range(n + 1)] for n, row in enumerate(triangle))
print("\nrow n equals the binomial coefficients C(n,k):", matches)

Pascal's triangle:
                 1             
               1   1           
             1   2   1         
           1   3   3   1       
         1   4   6   4   1     
       1   5  10  10   5   1   

row n equals the binomial coefficients C(n,k): True


In [5]:
import sympy as sp

x, y = sp.symbols("x y")
expanded = sp.expand((x + y) ** 4)
print("(x + y)^4 =", expanded)
print("coefficients:", [sp.binomial(4, k) for k in range(5)], "  <- this is Pascal's row 4")

(x + y)^4 = x**4 + 4*x**3*y + 6*x**2*y**2 + 4*x*y**3 + y**4
coefficients: [1, 4, 6, 4, 1]   <- this is Pascal's row 4


## Inclusion–Exclusion

When sets overlap, you cannot just add their sizes — you would double-count the overlap. The **inclusion–exclusion principle** for two sets is

$$|A \cup B| = |A| + |B| - |A \cap B|.$$

For example, how many integers in $1..100$ are divisible by $2$ **or** $3$?

In [6]:
N = 100
div2 = set(range(2, N + 1, 2))
div3 = set(range(3, N + 1, 3))
div6 = div2 & div3            # divisible by both 2 and 3  ==  divisible by 6

formula = len(div2) + len(div3) - len(div6)
print(f"|div by 2| = {len(div2)},  |div by 3| = {len(div3)},  |div by 6| = {len(div6)}")
print(f"inclusion-exclusion: {len(div2)} + {len(div3)} - {len(div6)} = {formula}")
print(f"brute-force |div2 union div3| = {len(div2 | div3)}")

|div by 2| = 50,  |div by 3| = 33,  |div by 6| = 16
inclusion-exclusion: 50 + 33 - 16 = 67
brute-force |div2 union div3| = 67


## The Pigeonhole Principle

The **pigeonhole principle** is simple but powerful: *if $n+1$ objects are placed into $n$ boxes, then at least one box contains two or more objects.* More generally, with $k$ objects in $n$ boxes, some box holds at least $\left\lceil k/n \right\rceil$ objects.

A classic consequence: in any group of $13$ people, at least two share a birth **month** (there are only $12$ months).

In [7]:
import math

people, months = 13, 12
print(f"{people} people, {months} months -> some month has at least "
      f"ceil({people}/{months}) = {math.ceil(people / months)} people. Guaranteed.")

# Demonstrate: no matter how you assign 13 people to 12 months, a month repeats.
from itertools import cycle
assignment = {}
for person, month in zip(range(people), cycle(range(months))):
    assignment.setdefault(month, []).append(person)
busiest = max(assignment.values(), key=len)
print(f"Example assignment: month with the most people has {len(busiest)} people:", busiest)

13 people, 12 months -> some month has at least ceil(13/12) = 2 people. Guaranteed.
Example assignment: month with the most people has 2 people: [0, 12]


## Worked Examples

A few examples solved with Python. (The exercises in the next section are for you to solve — their solutions live in the matching notebook in `solutions/`.)

### Example 1: Counting Passwords

By the product rule, a length-4 password of lowercase letters has $26^4$ possibilities. How many length-8 passwords use letters and digits ($62$ symbols)?

In [8]:
print("lowercase, length 4 (26^4):", 26 ** 4)
print("alphanumeric, length 8 (62^8):", 62 ** 8)

lowercase, length 4 (26^4): 456976
alphanumeric, length 8 (62^8): 218340105584896


### Example 2: A Row of Pascal's Triangle

Row $n$ of Pascal's triangle is $\binom{n}{0}, \binom{n}{1}, \dots, \binom{n}{n}$. Print row 5.

In [9]:
import math

print([math.comb(5, k) for k in range(6)])

[1, 5, 10, 10, 5, 1]


## Counting Practice Problems

Write your Python in the code cell beneath each problem (some starter code is provided). Worked solutions are in [`solutions/15_counting_key.ipynb`](solutions/15_counting_key.ipynb).

### Problem 1: Arrangements

In how many ways can 5 distinct books be arranged on a shelf? Compute $5!$ with `math.factorial`.

In [10]:
import math
# WRITE YOUR CODE BELOW


### Problem 2: Combinations

How many ways are there to choose a committee of 3 people from 10 (order does not matter)? Compute $\binom{10}{3}$ with `math.comb`.

In [11]:
import math
# WRITE YOUR CODE BELOW


### Problem 3: Permutations

How many ways are there to choose a president, vice-president, and secretary from 10 people (order matters)? Compute $P(10, 3)$ with `math.perm`.

In [12]:
import math
# WRITE YOUR CODE BELOW


### Problem 4: Inclusion-Exclusion

How many integers from 1 to 100 are divisible by 2 **or** 5? Use $\lfloor 100/2 \rfloor + \lfloor 100/5 \rfloor - \lfloor 100/10 \rfloor$.

In [13]:
# WRITE YOUR CODE BELOW
